In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-VZytJ5BN3ErM


In [2]:
# import v1.0
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# LLM 모델
# FewShotPromptTemplate
from langchain_core.prompts.few_shot import FewShotPromptTemplate

## FewShotPromptTemplate 설계

In [3]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        # ↓ 원하는 형식의 답변
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 도쿄
            언어: 일본어
            음식: 스시와 라멘
            통화: 엔
        """
    },
    {
        "country": "브라질에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 브라질리아
            언어: 포르투갈어
            음식: 슈하스코와 페이조아다
            통화: 헤알
        """
    },
    {
        "country": "이집트에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 카이로
            언어: 아랍어
            음식: 코샤리와 팔라펠
            통화: 이집트 파운드
        """
    }
]

## 2단계: 예시용 프롬프트 템플릿 정의

In [5]:
example_template = """
    Human: {country}
    AI: {answer}
"""

example_prompt = PromptTemplate.from_template(example_template)

example_prompt

PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='\n    Human: {country}\n    AI: {answer}\n')

## 3단계: FewShotPromptTemplate 생성 후 결합

In [6]:
prompt = FewShotPromptTemplate(
    # 질문
    example_prompt=example_prompt,
    # 예시
    examples=examples,
    # 사용자의 질문
    suffix="Human: {country}에 대해서 어떻게 알고 있어요?",
    input_variables=["country"]
)

In [8]:
prompt

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'country': '프랑스에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도: 파리\n            언어: 프랑스어\n            음식: 와인과 치즈\n            통화: 유로\n        '}, {'country': '일본에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도: 도쿄\n            언어: 일본어\n            음식: 스시와 라멘\n            통화: 엔\n        '}, {'country': '브라질에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도: 브라질리아\n            언어: 포르투갈어\n            음식: 슈하스코와 페이조아다\n            통화: 헤알\n        '}, {'country': '이집트에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도: 카이로\n            언어: 아랍어\n            음식: 코샤리와 팔라펠\n            통화: 이집트 파운드\n        '}], example_prompt=PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='\n    Human: {country}\n    AI: {answer}\n'), suffix='Human: {country}

In [58]:
chat = ChatOpenAI(temperature=0)

In [10]:
chain = prompt | chat

In [11]:
result = chain.invoke({
    "country": "한국"
})

In [14]:
print(result.content)

AI: 
            저는 이렇게 알고 있어요.
            수도: 서울
            언어: 한국어
            음식: 김치와 불고기
            통화: 대한민국 원


In [15]:
result = chain.invoke({
    "country": "영국"
})

In [19]:
print(result.content)

AI: 
            저는 이렇게 알고 있어요.
            수도: 런던
            언어: 영어
            음식: 피쉬 앤 칩스
            통화: 파운드


## FewShotChatMessagePromptTemplate 설계

In [18]:
# v1.0
# ChatModel
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain_core.prompts.chat import ChatPromptTemplate

In [31]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        # ↓ 원하는 형식의 답변
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 도쿄
            언어: 일본어
            음식: 스시와 라멘
            통화: 엔
        """
    },
    {
        "country": "브라질에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 브라질리아
            언어: 포르투갈어
            음식: 슈하스코와 페이조아다
            통화: 헤알
        """
    },
    {
        "country": "이집트에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 카이로
            언어: 아랍어
            음식: 코샤리와 팔라펠
            통화: 이집트 파운드
        """
    }
]

## 2단계 예시용 프롬프트 템플릿 정의

In [68]:
example = ChatPromptTemplate.from_messages([
    ("human", "{country}에 대해서 어떻게 알고 있어요?"),
    ("ai", "{answer}"),
])

In [70]:
example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example,
    examples=examples
)

In [79]:
final = prompt | chain

In [71]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 지리학 전문가입니다. 짧은 답변을 제공하세요. 예시 내용대로 답변하세요"),
    example_prompt,
    ("human", "{country}에 대해서 어떻게 알고 있어요?")
])

In [72]:
print(final_prompt)

input_variables=['country'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 지리학 전문가입니다. 짧은 답변을 제공하세요. 예시 내용대로 답변하세요'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'country': '프랑스에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도: 파리\n            언어: 프랑스어\n            음식: 와인과 치즈\n            통화: 유로\n        '}, {'country': '일본에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도: 도쿄\n            언어: 일본어\n            음식: 스시와 라멘\n            통화: 엔\n        '}, {'country': '브라질에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도: 브라질리아\n            언어: 포르투갈어\n            음식: 슈하스코와 페이조아다\n            통화: 헤알\n        '}, {'country': '이집트에 대해서 어떻게 알고 있나요?', 'answer': '\n            저는 이렇게 알고 있어요.\n            수도: 카이로\n            언어: 아랍어\n            음식: 코샤리와 팔라펠\n            통화: 이집트

In [76]:
final_chain = final_prompt | chat

In [77]:
result = final_chain.invoke({
    "country": "일본"
})

In [81]:
print(result.content)

일본은 아시아 대륙 동쪽에 위치한 섬나라로, 수도는 도쿄이며 일본어를 사용합니다. 일본은 문화, 기술, 엔터테인먼트 등 다양한 분야에서 세계적으로 유명하며 스시, 라멘, 차 등의 전통 음식이 유명합니다. 또한 일본의 자연경관과 역사적인 유적지도 많이 알려져 있습니다. 일본은 엔을 통화로 사용하고 있습니다.


## LengthBasedExampleSelector 설계

In [82]:
# v1.0
from langchain_core.example_selectors.length_based import LengthBasedExampleSelector

In [83]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        # ↓ 원하는 형식의 답변
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 도쿄
            언어: 일본어
            음식: 스시와 라멘
            통화: 엔
        """
    },
    {
        "country": "브라질에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 브라질리아
            언어: 포르투갈어
            음식: 슈하스코와 페이조아다
            통화: 헤알
        """
    },
    {
        "country": "이집트에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 카이로
            언어: 아랍어
            음식: 코샤리와 팔라펠
            통화: 이집트 파운드
        """
    }
]

In [84]:
example_prompt = PromptTemplate.from_template("Human: {country}\nAI: {answer}")

## Selector 연결

In [102]:
example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    # 예제의 양 허용 (토큰 개수)
    max_length=200
)

In [103]:
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: {country}에 대해서 어떻게 알고 있나요?",
    input_variables=["country"]
)

In [105]:
print(prompt.format(**{
    "country": "브라질"
}))

Human: 프랑스에 대해서 어떻게 알고 있나요?
AI: 
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        

Human: 일본에 대해서 어떻게 알고 있나요?
AI: 
            저는 이렇게 알고 있어요.
            수도: 도쿄
            언어: 일본어
            음식: 스시와 라멘
            통화: 엔
        

Human: 브라질에 대해서 어떻게 알고 있나요?


In [106]:
final_chain = prompt | chat

In [107]:
result = final_chain.invoke({
    "country": "독일"
})

In [109]:
print(result.content)

AI: 
            저는 이렇게 알고 있어요.
            수도: 베를린
            언어: 독일어
            음식: 소세지와 맥주
            통화: 유로


## Chat 모델(LengthBasedExampleSelector)

In [111]:
examples = [
    {
        "country": "프랑스에 대해서 어떻게 알고 있나요?",
        # ↓ 원하는 형식의 답변
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 파리
            언어: 프랑스어
            음식: 와인과 치즈
            통화: 유로
        """
    },
    {
        "country": "일본에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 도쿄
            언어: 일본어
            음식: 스시와 라멘
            통화: 엔
        """
    },
    {
        "country": "브라질에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 브라질리아
            언어: 포르투갈어
            음식: 슈하스코와 페이조아다
            통화: 헤알
        """
    },
    {
        "country": "이집트에 대해서 어떻게 알고 있나요?",
        "answer": """
            저는 이렇게 알고 있어요.
            수도: 카이로
            언어: 아랍어
            음식: 코샤리와 팔라펠
            통화: 이집트 파운드
        """
    }
]

In [122]:
length_prompt = PromptTemplate(
    input_variables=["country", "answer"],
    template="""
        Human: {country}에 대해서 어떻게 알고 있어요? \n
        AI: {answer}
    """
)

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{country}에 대해서 어떻게 알고 있어요?"),
    ("ai", "{answer}"),
])

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=length_prompt,
    max_length=200
)

fewshot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt
)

In [123]:
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 지리학 전문가입니다."),
    fewshot_prompt,
    ("human", "{country}에 대해 어떻게 알고 있어요?"),
])

In [124]:
chain = final_prompt | chat

In [125]:
result = chain.invoke({
    "country": "대한민국"
})

In [128]:
print(result.content)


            저는 이렇게 알고 있어요.
            수도: 서울
            언어: 한국어
            음식: 김치, 불고기
            통화: 대한민국 원
